# UPI Transaction Failure Prediction (Explainable Model)

This notebook trains machine learning models to predict UPI transaction failures and also explains **why a transaction may fail**.

Dataset: Synthetic UPI transaction dataset

Pipeline:
1. Load dataset
2. Feature engineering
3. Data preprocessing
4. Train/Test split
5. Model training (Random Forest & XGBoost)
6. Model evaluation
7. Feature importance analysis
8. Explainable prediction demo


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


## Load Dataset

In [2]:
df = pd.read_csv('../data/transactions.csv')
df.head()

,Transaction ID,Timestamp,Sender Name,Sender UPI ID,Receiver Name,Receiver UPI ID,Amount (INR),Status
0,4d3db980-46cd-4158-a812-dcb77055d0d2,2024-06-22 04:06:38,Tiya Mall,4161803452@okaxis,Mohanlal Golla,7776849307@okybl,3907.34,FAILED
1,099ee548-2fc1-4811-bf92-559c467ca792,2024-06-19 06:04:49,Mohanlal Bakshi,8908837379@okaxis,Mehul Sankaran,7683454560@okaxis,8404.55,SUCCESS
2,d4c05732-6b1b-4bab-90b9-efe09d252b99,2024-06-04 04:56:09,Kismat Bora,4633654150@okybl,Diya Goel,2598130823@okicici,941.88,SUCCESS
3,e8df92ee-8b04-4133-af5a-5f412180c8ab,2024-06-09 09:56:07,Ayesha Korpal,7018842771@okhdfcbank,Rhea Kothari,2246623650@okaxis,8926.00,SUCCESS
4,e7d675d3-04f1-419c-a841-7a04662560b7,2024-06-25 08:38:19,Jivin Batta,1977143985@okybl,Baiju Issac,5245672729@okybl,2800.55,SUCCESS


## Feature Engineering

In [3]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

## Extract Bank Identifiers

In [4]:
df['Sender_Bank'] = df['Sender UPI ID'].apply(lambda x: x.split('@')[1])
df['Receiver_Bank'] = df['Receiver UPI ID'].apply(lambda x: x.split('@')[1])

## Encode Categorical Features

In [5]:
le = LabelEncoder()
df['Sender_Bank'] = le.fit_transform(df['Sender_Bank'])
df['Receiver_Bank'] = le.fit_transform(df['Receiver_Bank'])
df['Status'] = df['Status'].map({'SUCCESS':0,'FAILED':1})

## Feature Selection

In [6]:
features = ['Amount (INR)','Hour','DayOfWeek','Sender_Bank','Receiver_Bank']
X = df[features]
y = df['Status']

## Train Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training samples:', X_train.shape)
print('Testing samples:', X_test.shape)

Training samples: (800, 5)
Testing samples: (200, 5)


## Random Forest Training

In [8]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print('Random Forest Accuracy:', accuracy_score(y_test, rf_preds))

Random Forest Accuracy: 0.57


## XGBoost Training

In [13]:
xgb = XGBClassifier( eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
print('XGBoost Accuracy:', accuracy_score(y_test, xgb_preds))

XGBoost Accuracy: 0.535


## Model Evaluation

In [10]:
print(classification_report(y_test, xgb_preds))
prob = xgb.predict_proba(X_test)[:,1]
print('ROC AUC Score:', roc_auc_score(y_test, prob))

              precision    recall  f1-score   support

           0       0.55      0.56      0.56       103
           1       0.52      0.51      0.51        97

    accuracy                           0.54       200
   macro avg       0.53      0.53      0.53       200
weighted avg       0.53      0.54      0.53       200

ROC AUC Score: 0.5428885997397658


## Feature Importance (Reason Analysis)

In [11]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance': xgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance

,Feature,Importance
4,Receiver_Bank,0.215015
2,DayOfWeek,0.208193
1,Hour,0.197325
3,Sender_Bank,0.194648
0,Amount (INR),0.184819


## Explainable Prediction

The following prediction demonstrates how the model evaluates a new transaction and identifies the main contributing factors.

In [12]:
sample_transaction = pd.DataFrame({
    'Amount (INR)': [3500],
    'Hour': [21],
    'DayOfWeek': [5],
    'Sender_Bank': [2],
    'Receiver_Bank': [1]
})

failure_prob = xgb.predict_proba(sample_transaction)[0][1]

print('Predicted Failure Probability:', round(failure_prob,2))

top_features = importance.head(3)['Feature'].tolist()

print('\nTop factors influencing prediction:')
for f in top_features:
    print('-', f)

if failure_prob > 0.6:
    print('\n⚠ High chance of transaction failure due to these contributing factors.')
else:
    print('\n✅ Transaction likely to succeed based on current feature values.')

Predicted Failure Probability: 0.27

Top factors influencing prediction:
- Receiver_Bank
- DayOfWeek
- Hour

✅ Transaction likely to succeed based on current feature values.
